# [Chapter 19 — HIV, §19.3] Bipartite HIV with Treatment-as-Prevention

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 19 — HIV
**Considerations developed:** 4 (force-of-infection direction), 7 (mixing balance), 12 (basic vs effective $R$)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_19_hiv/01_HIV_bipartite_with_treatment.ipynb)


## What this notebook does

Loads the synthetic 30-year HIV prevalence dataset (M and F annual estimates plus treatment coverage rollout) and fits a bipartite model with three compartments per sex: susceptible (S), untreated infectious (I), and treated/suppressed (T, near-zero infectiousness). Demonstrates how treatment-as-prevention modifies the *effective* reproductive number even when contact patterns are unchanged — a Consideration~12 illustration.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_19
from shared.verification import assert_within_tolerance
set_seed_chapter_19()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Load HIV surveillance data

In [ ]:
with open(os.path.join(DATA_ROOT, 'hiv', 'annual_prevalence_with_treatment.csv')) as f:
    next(f)
    rows = [(float(y), float(pM), float(pF), float(cov)) for y, pM, pF, cov in csv.reader(f)]
year = np.array([r[0] for r in rows])
prev_M_obs = np.array([r[1] for r in rows])
prev_F_obs = np.array([r[2] for r in rows])
tx_cov = np.array([r[3] for r in rows])

with open(os.path.join(DATA_ROOT, 'hiv', 'metadata.json')) as f:
    truth = json.load(f)['generating_parameters_TRUTH_FOR_NOTEBOOK_VERIFICATION']
print(f'Data: {len(year)} annual snapshots, {year[0]:.0f} to {year[-1]:.0f}')
print(f'Treatment rollout: {truth["treatment_rollout_start_year"]} to {truth["treatment_rollout_end_year"]}')
print(f'Maximum treatment coverage: {truth["treatment_coverage_max"]:.0%}')


## Visualize prevalence and treatment rollout

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(year, prev_M_obs*100, 'o-', color=BOOK_COLORS['primary'], lw=1.5, label='Male prevalence')
ax1.plot(year, prev_F_obs*100, 's-', color=BOOK_COLORS['secondary'], lw=1.5, label='Female prevalence')
ax1.set_xlabel('year')
ax1.set_ylabel('HIV prevalence (%)')
ax2 = ax1.twinx()
ax2.plot(year, tx_cov*100, '--', color='gray', lw=1.2, label='ART coverage')
ax2.set_ylabel('ART coverage (%)', color='gray')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.set_title('Synthetic generalized HIV epidemic with TasP rollout')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Fit the bipartite model with treatment compartment

In [ ]:
from shared.parameters import baseline_chapter_19
p = baseline_chapter_19()
tau_R, tau_R_treated = p['tau_R'], p['tau_R_treated']
NM, NF = p['NM'], p['NF']

def cov_at(t):
    if t < 10: return 0.0
    if t > 25: return 0.6
    return 0.6*(t-10)/15.0

def hiv_rhs(y, t, betaMF, betaFM, cMF, cFM):
    SM, IM, TM, SF, IF, TF = y
    eff_M = IM + TM*(tau_R_treated/tau_R)
    eff_F = IF + TF*(tau_R_treated/tau_R)
    inc_M = betaMF * cMF * SM * eff_F / NF
    inc_F = betaFM * cFM * SF * eff_M / NM
    rate_to_tx = cov_at(t) / 2.0
    return [-inc_M,
            inc_M - IM/tau_R - rate_to_tx*IM,
            rate_to_tx*IM - TM/tau_R_treated/2,
            -inc_F,
            inc_F - IF/tau_R - rate_to_tx*IF,
            rate_to_tx*IF - TF/tau_R_treated/2]

def predict(theta):
    betaMF, betaFM = theta
    y0 = [0.495, 0.005, 0.0, 0.495, 0.005, 0.0]
    t = np.linspace(0, 30, 361)
    sol = odeint(hiv_rhs, y0, t, args=(float(betaMF), float(betaFM), p['cMF'], p['cFM']))
    SM, IM, TM, SF, IF, TF = sol.T
    pred_M = (IM + TM)[::12] / NM
    pred_F = (IF + TF)[::12] / NF
    return pred_M, pred_F

def loss(theta):
    if theta[0] <= 0 or theta[1] <= 0: return 1e10
    pM, pF = predict(theta)
    return float(np.sum((pM - prev_M_obs)**2 + (pF - prev_F_obs)**2))

res = minimize(loss, x0=[0.15, 0.07], method='Nelder-Mead')
betaMF_hat, betaFM_hat = float(res.x[0]), float(res.x[1])
print(f'Fitted betaMF = {betaMF_hat:.3f}  (truth: {truth["betaMF"]})')
print(f'Fitted betaFM = {betaFM_hat:.3f}  (truth: {truth["betaFM"]})')
import math
R0_fit = math.sqrt(p['cMF']*p['cFM']*betaMF_hat*betaFM_hat*tau_R**2)
print(f'\nR_0 (untreated, fitted) = {R0_fit:.2f}  (truth: {truth["R0_untreated"]})')


## Fit visualization

In [ ]:
pM_fit, pF_fit = predict([betaMF_hat, betaFM_hat])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(year, prev_M_obs*100, 'o', color=BOOK_COLORS['primary'], ms=5, label='M data')
ax.plot(year, prev_F_obs*100, 's', color=BOOK_COLORS['secondary'], ms=5, label='F data')
ax.plot(year, pM_fit*100, '-', color=BOOK_COLORS['primary'], lw=1.5, label='M fit')
ax.plot(year, pF_fit*100, '-', color=BOOK_COLORS['secondary'], lw=1.5, label='F fit')
ax.set_xlabel('year')
ax.set_ylabel('prevalence (%)')
ax.set_title('Bipartite HIV fit with treatment-as-prevention')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verify both betas within 30% of truth
for label, est, true in [('betaMF', betaMF_hat, truth['betaMF']),
                          ('betaFM', betaFM_hat, truth['betaFM'])]:
    err = abs(est - true)/true
    assert err < 0.30, f'{label} off by {err:.0%}'
print('\nVerified: both transmission probabilities recovered within 30% of truth.')


## What's next

- Notebook 02 compares this generalized-epidemic profile with a concentrated-epidemic profile (e.g., MSM-only) to illustrate Consideration 7's mixing-balance dependence.
- Notebook 03 adds vertical (mother-to-child) transmission with intervention coverage, building on Chapter 15's vertical-transmission framework.
- The treatment compartment is a simplified TasP model; real models include CD4-staged disease progression and pre-exposure prophylaxis (PrEP) for at-risk uninfected populations.